In [1]:
import numpy as np
import pandas as pd
import glob, os, sys, subprocess
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import scipy.stats as st
import statsmodels.stats.api as sm

import Bio.PDB
from Bio import Seq, SeqIO
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.DSSP import make_dssp_dict
from Bio.PDB.Polypeptide import protein_letters_3to1

h37Rv = SeqIO.read("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/GCF_000195955.2_ASM19595v2_genomic.gbff", "genbank")
h37Rv_genes = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/mycobrowser_h37rv_genes_v4.csv")

os.chdir("../")
sys.path.append("utils")
from data_utils import *
from inSilicoMut_utils import *

who_variants = pd.read_csv("./data_processing/data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)
silent_lst = ['synonymous_variant', 'initiator_codon_variant', 'stop_retained_variant']

# Make distance maps

## Notes for AlphaFold structures:

Code to save only the high confidence coordinates

```
select high_confidence, b > 70
save ethA_alphaFold_highConf.pdb, high_confidence
```

<!-- 
<ul>
    <li></li>
</ul> -->

In [2]:
# # need to leave the Unnamed: 0 index column (don't save with index = False) because evcouplings.compare.distances.py reads in the dataframe with index_col = 0
# pncA_structure_coords = np.load("distance_maps/I6XD65.npy")
# pncA_distance_map = pd.read_csv("distance_maps/I6XD65.csv")

# katG_structure_coords = np.load("distance_maps/P9WIE5.npy")
# katG_distance_map = pd.read_csv("distance_maps/P9WIE5.csv")

# # check that this is a pairwise matrix, meaning that it's symmetric
# assert scipy.linalg.issymmetric(pncA_structure_coords)
# assert scipy.linalg.issymmetric(katG_structure_coords)

# # and also that the diagonals are all 0
# assert sum(np.diagonal(pd.DataFrame(pncA_structure_coords))) == 0
# assert sum(np.diagonal(pd.DataFrame(katG_structure_coords))) == 0

In [3]:
# Function to parse CIF file and extract necessary information
def extract_cif_info(cif_file):
    parser = MMCIFParser()
    structure = parser.get_structure('protein', cif_file)
    
    model = structure[0]
    
    # List to hold extracted information
    data = []
    
    # Extract information for each residue
    for chain in model:
        chain_id = chain.id
        for i, res in enumerate(chain):
            if res.id[0] == ' ':  # Exclude heteroatoms for now
                res_id = res.id[1]
                seqres_id = i + 1
                res_name = res.resname
                try:
                    one_letter_code = protein_letters_3to1[res_name]
                except KeyError:
                    one_letter_code = 'X'  # Unknown residue
                
                hetatm = res.id[0] != ' '
                coord = res['CA'].coord if 'CA' in res else None

                # # Get secondary structure assignment from DSSP -- not necessary for spatial clustering, and need to install additional dependencies, so skip for now
                # dssp_key = (chain_id, (' ', res_id, ' '))
                # if dssp_key in dssp_dict:
                #     sec_struct = dssp_dict[dssp_key][1]
                #     sec_struct_3state = 'H' if sec_struct in 'GHI' else 'E' if sec_struct == 'E' else 'C'
                # else:
                #     sec_struct = 'NA'
                #     sec_struct_3state = 'NA'
                sec_struct = 'NA'
                sec_struct_3state = 'NA'

                # chain index = 0 because there is only one chain
                # add 1 to len(data) to make it 1-indexed (in residue coordinate space, not index)
                data.append([
                    len(data) + 1, seqres_id, res_id, one_letter_code, res_name,
                    0, chain_id, sec_struct, sec_struct_3state, hetatm, coord
                ])
    
    # Create DataFrame
    columns = ['id', 'seqres_id', 'coord_id', 'one_letter_code',
               'three_letter_code', 'chain_index', 'chain_id', 'sec_struct',
               'sec_struct_3state', 'hetatm', 'coord']
    df = pd.DataFrame(data, columns=columns)
    return df

In [3]:
# mmcif_file = 'ethA_AlphaFold.cif'

# df_ethA_AF = extract_cif_info(mmcif_file)

# # residues 1 and 484-489 are low-confidence in alpha fold, so exclude
# df_ethA_AF_highConf = df_ethA_AF.query("id >= 2 & id <= 483").reset_index(drop=True)
# df_ethA_AF_highConf.to_csv("distance_maps/P9WNF9_AF.csv")

In [5]:
RNA_polymerase = extract_cif_info("spatial_clustering/PDB/5uhb.cif")

In [6]:
DNA_gyrase = extract_cif_info("spatial_clustering/PDB/5bs8.cif")

In [124]:
# A, C are gyrA; B, D are gyrB. E-H are DNA substrates
print(DNA_gyrase.chain_id.unique())

gyrA = DNA_gyrase.query("chain_id in ['A', 'C']")
gyrB = DNA_gyrase.query("chain_id in ['B', 'D']")

# chains A and C have the same residues represented
# chain D has two more residues than chain B, so take D
gyrA = gyrA.query("chain_id=='A'")
gyrB = gyrB.query("chain_id=='D'")

assert len(gyrA) == gyrA.coord_id.nunique()
assert len(gyrB) == gyrB.coord_id.nunique()

['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H']


In [125]:
gyrB.query("coord_id in [674, 675]")

,id,seqres_id,coord_id,one_letter_code,three_letter_code,chain_index,chain_id,sec_struct,sec_struct_3state,hetatm,coord
1462,1463,246,674,D,ASP,0,D,NA,NA,False,"[37.362, -23.997, 45.799]"
1463,1464,247,675,V,VAL,0,D,NA,NA,False,"[34.732, -26.002, 47.704]"


In [96]:
# some sequence inconsistencies, but it's only 3 residues at the end, so maybe some random mutation or poor sequencing happened. Just exclude them
gyrA_test_seq = dict(zip(gyrA['coord_id'], gyrA['one_letter_code']))
gyrB_test_seq = dict(zip(gyrB['coord_id'], gyrB['one_letter_code']))

In [106]:
print(min(list(gyrA_test_seq.keys())), max(list(gyrA_test_seq.keys())))

for pos, residue in gyrA_test_seq.items():
    if residue != gyrA_protein_seq[pos-1]:
        print(pos, residue, gyrA_protein_seq[pos-1])

15 501
501 I A


In [107]:
print(min(list(gyrB_test_seq.keys())), max(list(gyrB_test_seq.keys())))

for pos, residue in gyrB_test_seq.items():
    if residue != gyrB_protein_seq[pos-1]:
        print(pos, residue, gyrB_protein_seq[pos-1])

424 675
424 N R
425 A E


In [118]:
gyrA = gyrA.query("coord_id != 501")
gyrA['id'] = gyrA['coord_id']

# rename these natural numbers
gyrA['seqres_id'] = np.arange(1, len(gyrA)+1)

gyrA.to_csv("distance_maps/5BS8_gyrA.csv")
get_distance_map_coordinates(gyrA, "5BS8_gyrA")

(485, 485)


In [119]:
gyrB = gyrB.query("coord_id not in [424, 425]")
gyrB['id'] = gyrB['coord_id']

# rename these natural numbers
gyrB['seqres_id'] = np.arange(1, len(gyrB)+1)
gyrB.to_csv("distance_maps/5BS8_gyrB.csv")
get_distance_map_coordinates(gyrB, "5BS8_gyrB")

(245, 245)


In [24]:
# from PDB, chain C = rpoB, but the number is slightly off too
# residue 7 in PDB is residue 1 in H37Rv, so subtract 6
rpoB = RNA_polymerase.query("chain_id=='C'").reset_index(drop=True)
rpoB['coord_id'] -= 6
rpoB['id'] = rpoB['coord_id']

In [25]:
# did some manual checks
rpoB_protein_seq = h37Rv.seq[759807-1:763325].translate()


In [29]:
rpoB_protein_seq[21:1147]

Seq('SNNSVPGAPNRVSFAKLREPLEVPGLLDVQTDSFEWLIGSPRWRESAAERGDVN...EDE')

In [82]:
# from PDB, chain C = rpoB, but the number is slightly off too
# residue 7 in PDB is residue 1 in H37Rv, so subtract 6
rpoB = RNA_polymerase.query("chain_id=='C'").reset_index(drop=True)
rpoB['coord_id'] -= 6
rpoB['id'] = rpoB['coord_id']

rpoB.to_csv("distance_maps/5UHB.csv")

# did some manual checks
rpoB_protein_seq = h37Rv.seq[759807-1:763325].translate()
gyrA_protein_seq = h37Rv.seq[7301:9818].translate()
gyrB_protein_seq = h37Rv.seq[5239:7267].translate()

assert rpoB_protein_seq[-1] == '*'
assert gyrA_protein_seq[-1] == '*'
assert gyrB_protein_seq[-1] == '*'

In [33]:
get_distance_map_coordinates(rpoB, "5UHB")

(1126, 1126)


In [11]:
def calculate_pairwise_distances(coords):
    coords = np.array([coord for coord in coords if coord is not None])
    distances = np.linalg.norm(coords[:, np.newaxis] - coords, axis=-1)
    return distances

In [15]:
def get_distance_map_coordinates(df, dm_name):
    
    df.to_csv(f"./spatial_clustering/distance_maps/{dm_name}.csv")

    assert sum(pd.isnull(df['coord'])) == 0
    
    # the coordinates are for the alpha carbon atom in each residue
    ca_coords = list(df['coord'])
    ca_distances = calculate_pairwise_distances(ca_coords)
    
    np.save(f"./spatial_clustering/distance_maps/{dm_name}.npy", ca_distances)
    
    # pairwise matrix check
    assert scipy.linalg.issymmetric(ca_distances)
    assert sum(np.diagonal(pd.DataFrame(ca_distances))) == 0
    
    print(ca_distances.shape)

In [34]:
Rv0678 = extract_cif_info("spatial_clustering/PDB/4nb5.cif")

# get_distance_map_coordinates(Rv0678, "4nb5_Rv0678")

In [37]:
Rv0678['coord_id'].max()

163

In [38]:
Rv0678['id'] = Rv0678['chain_id'] + '_' + Rv0678['coord_id'].astype(str)

In [39]:
get_distance_map_coordinates(Rv0678, "4nb5_Rv0678")

(587, 587)


In [150]:
katG = extract_cif_info("spatial_clustering/PDB/4c51.cif")

In [151]:
katG.chain_id.unique()

array(['A', 'B'], dtype=object)

In [152]:
katG['id'] = katG['chain_id'] + '_' + katG['coord_id'].astype(str)

In [154]:
katG.coord_id.nunique()

717

In [155]:
katG.chain_id.unique()

array(['A', 'B'], dtype=object)

In [156]:
get_distance_map_coordinates(katG, "4c51_katG")

(1434, 1434)


# Results of Clustering on Averaged Fold Change MIC Predictions from Site-Saturation Mutagenesis

In [54]:
def get_significant_GeO_scores(gene, pval_thresh=0.05):

    residue_data = pd.read_csv(f"spatial_clustering/{gene}/values_to_cluster.csv")
    
    GeO_scores = pd.read_csv(f"spatial_clustering/{gene}/G_scores.csv").merge(residue_data, on=['residue', 'chain', 'coordinate'])

    # these are GeO scores for each residue after shuffling the residues
    # perform a permutation test to see if the GeO score for each residue is significantly different from the the null
    GeO_permutation_results = pd.read_csv(f"spatial_clustering/{gene}/random_GeO_iterations_10000.csv.gz", compression="gzip", index_col=[0])
    
    for i, row in GeO_scores.iterrows():
    
        residue = row['residue']
        GeO_score = row['G_score']
    
        # what proportion of the permuted Getis-Ord statistics are at least as extreme as the Getis-Ord statistic for a given residue
        # positive GeO score is hot spot (R- or S-associated, depending on the prefix), negative GeO score is cold spot (neutral mutations)
        if GeO_score > 0:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values >= GeO_score)
        else:
            pvalue = np.mean(GeO_permutation_results.loc[residue, :].values <= GeO_score)
    
        GeO_scores.loc[i, "pval"] = pvalue

    _, bh_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='fdr_bh', is_sorted=False, returnsorted=False)
    _, bonferroni_pvals, _, _ = sm.multipletests(GeO_scores["pval"], method='bonferroni', is_sorted=False, returnsorted=False)
    
    GeO_scores['BH_pval'] = bh_pvals
    GeO_scores['Bonferroni_pval'] = bonferroni_pvals
    
    # GeO_scores.loc[(GeO_scores['BH_pval'] <= pval_thresh) & (GeO_scores['G_score'] > 0) & (GeO_scores['average'] > 0), 'clustering_result'] = 1
    GeO_scores.loc[(GeO_scores['BH_pval'] <= pval_thresh) & (GeO_scores['G_score'] > 0), 'clustering_result'] = 1

    # significant negative G score indicates cold spot (clustering of low values)
    GeO_scores.loc[(GeO_scores['BH_pval'] <= pval_thresh) & (GeO_scores['G_score'] < 0) & (GeO_scores['average'] < 0), 'clustering_result'] = -1

    # significant negative G score indicates cold spot (clustering of low values), but if the average prediction for that residue is positive, then it's clustering of neutral mutations
    # GeO_scores.loc[(GeO_scores['BH_pval'] <= pval_thresh) & (GeO_scores['G_score'] < 0) & (GeO_scores['average'] > 0), 'clustering_result'] = 0
    
    # exclude the first and last residue because those are probably false positives due to it being easier to get a significant result when there are fewer residues around
    # GeO_scores.loc[GeO_scores['residue']==np.min(GeO_scores['residue']), 'clustering_result'] = np.nan
    # GeO_scores.loc[GeO_scores['residue']==np.max(GeO_scores['residue']), 'clustering_result'] = np.nan
        
    print(GeO_scores['clustering_result'].value_counts())

    # save and return it for future use
    GeO_scores.to_csv(f"./supplement/{gene}_GeO_scores.csv", index=False)
    
    return GeO_scores

In [55]:
gene = 'Rv0678'
df_Rv0678 = get_significant_GeO_scores(gene)

clustering_result
 1.0    25
-1.0     3
Name: count, dtype: int64


In [23]:
catalytic_triad = [8, 96, 138]
iron_coordinating = [49, 51, 57, 71]

gene = 'pncA'
df_pncA = get_significant_GeO_scores(gene)

clustering_result
 1.0    14
-1.0     7
Name: count, dtype: int64


In [70]:
df_pncA.query("residue in @catalytic_triad")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,clustering_result
6,8,2.519451,2.212396,0.0133,0.074141,1.0000,NaN
94,96,2.544833,2.048351,0.0137,0.074141,1.0000,NaN
136,138,3.712178,0.517343,0.0009,0.012738,0.1656,1.0


In [71]:
df_pncA.query("residue in @iron_coordinating")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,clustering_result
47,49,3.048002,2.206316,0.0044,0.036800,0.8096,1.0
49,51,1.746432,2.988830,0.0587,0.159121,1.0000,NaN
55,57,2.127011,3.597437,0.0310,0.107623,1.0000,NaN
69,71,1.554132,0.557121,0.0810,0.177429,1.0000,NaN


In [24]:
# Arg104, Trp107, and His108 in a pocket distal to the heme, and His270, Trp321, and Asp381 in a pocket proximal to the heme. A covalently linked “MYW catalytic triad” is formed by the conserved residues, Met255, Tyr229, and Trp107

gene = 'katG'
df_katG = get_significant_GeO_scores(gene)

clustering_result
 1.0    97
-1.0    70
Name: count, dtype: int64


In [25]:
gene = 'ethA'
df_ethA = get_significant_GeO_scores(gene)

clustering_result
1.0    5
Name: count, dtype: int64


In [26]:
df_ethA.query("clustering_result in [1, -1]")

,residue,G_score,average,pval,BH_pval,Bonferroni_pval,clustering_result
53,55,4.605937,0.240914,0.0005,0.0482,0.2410,1.0
55,57,4.559195,1.105226,0.0004,0.0482,0.1928,1.0
57,59,4.372211,0.132343,0.0004,0.0482,0.1928,1.0
61,63,4.223945,-0.027199,0.0005,0.0482,0.2410,1.0
291,293,5.634349,0.129163,0.0001,0.0482,0.0482,1.0


From this link: https://www.uniprot.org/uniprotkb/P9WNF9/entry

<ul>
    <li>FAD binding sites: 15, 36, 44-47, 56, 104</li>
    <li>NADP+ binding sites: 54-56, 183-189, 207-208</li>
    <li>Transition state stabilizer: 292</li>
</ul>

In [6]:
if 'annotation' in df_ethA.columns:
    del df_ethA['annotation']

# predicted binding site
FAD_binding_sites = [15, 36, 44, 45, 46, 47, 56, 104]
NADP_binding_sites = [54, 55, 56, 183, 184, 185, 186, 187, 188, 189, 207, 208]
# transition_state_stabilizer = [292]

df_ethA.loc[df_ethA['residue'].isin(FAD_binding_sites), 'annotation'] = 'FAD binding'
df_ethA.loc[df_ethA['residue'].isin(NADP_binding_sites), 'annotation'] = 'NADP+ binding'
# df_ethA.loc[df_ethA['residue'].isin(transition_state_stabilizer), 'annotation'] = 'transition state stabilizer'
df_ethA['annotation'] = df_ethA['annotation'].replace('nan', np.nan)

len(df_ethA.query("(residue in @FAD_binding_sites | residue in @NADP_binding_sites) & BH_pval <= 0.1").sort_values("G_score", ascending=False)[['residue', 'G_score', 'annotation', 'BH_pval']]), len(df_ethA.dropna(subset='annotation'))

(7, 19)

In [7]:
df_ethA.annotation.value_counts()

annotation
NADP+ binding    12
FAD binding       7
Name: count, dtype: int64

In [8]:
pos_string = '+'.join(np.array(list(set(FAD_binding_sites).union(NADP_binding_sites))).astype(str))
print(f"select binding_sites, resi {pos_string}")
print(f"color yellow, binding_sites")

select binding_sites, resi 15+36+44+45+46+47+54+55+56+185+183+186+187+188+189+184+207+208+104
color yellow, binding_sites


In [17]:
intersect_pos_string = '+'.join(np.array(list(set(FAD_binding_sites).union(NADP_binding_sites).intersection(df_ethA.query("clustering_result==1").residue))).astype(str))
print(f"select binding_sites_hot_spots, resi {intersect_pos_string}")
print(f"color orange, binding_sites_hot_spots")

select binding_sites_hot_spots, resi 55
color orange, binding_sites_hot_spots


In [27]:
# moxifloxacin. QRDR in gyrA: 74-113, gyrB: 461-499
df_gyrA = get_significant_GeO_scores('gyrA')
df_gyrB = get_significant_GeO_scores('gyrB')

clustering_result
-1.0    84
 1.0    38
Name: count, dtype: int64
clustering_result
-1.0    28
 1.0    17
Name: count, dtype: int64


In [28]:
len(df_gyrA.query("residue >= 74 & residue <= 113")), len(df_gyrA.query("residue >= 74 & residue <= 113 & clustering_result==1"))

(40, 26)

In [29]:
len(df_gyrB.query("residue >= 461 & residue <= 499")), len(df_gyrB.query("residue >= 461 & residue <= 499 & clustering_result==1"))

(39, 9)

In [30]:
df_rpoB = get_significant_GeO_scores('rpoB')

clustering_result
1.0    117
Name: count, dtype: int64


In [64]:
# how many of the 27 RRDR sites are hot spots
print(len(df_rpoB.query("clustering_result==1")))

# There are 27 RRDR residues
print(len(df_rpoB.query("clustering_result==1 & residue >= 426 & residue <= 452")))

117
27


In [80]:
df_rpoB.query("clustering_result==1").residue.min(), df_rpoB.query("clustering_result==1").residue.max()

(47, 674)

In [70]:
def print_pymol_selection_commands(df, coord_adj=0, name_prefix='', R_color='firebrick', S_color='marine', neutral_color='lightorange'):
    '''
    name_prefix is for genes whose protein products form a complex, i.e. gyrA and gyrB. Add the gene name prefix so that there are different selections in PyMol.
    '''
    
    pval_col = 'BH_pval'
    
    # reset the coloring to gray
    print("select all")
    print("color gray80, all\n")

    for chain_name in df.chain.unique():
        R_hot_spots = [str(num + coord_adj) for num in df.query("chain==@chain_name & clustering_result==1").coordinate.values]
        R_hot_spots = '+'.join(R_hot_spots)
        
        if len(R_hot_spots) > 0:
            print(f"select {name_prefix}R_hot_spots, resi {R_hot_spots} and chain {chain_name}")
            print(f"color {R_color}, {name_prefix}R_hot_spots\n")

        S_hot_spots = [str(num + coord_adj) for num in df.query("chain==@chain_name & clustering_result==-1").coordinate.values]
        S_hot_spots = '+'.join(S_hot_spots)
        
        if len(S_hot_spots) > 0:
            print(f"select {name_prefix}S_hot_spots, resi {S_hot_spots} and chain {chain_name}")
            print(f"color {S_color}, {name_prefix}S_hot_spots\n")

In [138]:
print_pymol_selection_commands(df_Rv0678, coord_adj=2)

select all
color gray80, all

select R_hot_spots, resi 67+68+69+70+71+73 and chain A
color firebrick, R_hot_spots

select S_hot_spots, resi 146 and chain A
color marine, S_hot_spots

select R_hot_spots, resi 67+68+69+70+71+73 and chain B
color firebrick, R_hot_spots

select R_hot_spots, resi 67+68+69+70+71+73 and chain C
color firebrick, R_hot_spots

select S_hot_spots, resi 146 and chain C
color marine, S_hot_spots

select R_hot_spots, resi 66+67+68+69+70+71+73 and chain D
color firebrick, R_hot_spots

select S_hot_spots, resi 146 and chain D
color marine, S_hot_spots



In [74]:
print_pymol_selection_commands(df_pncA)

select all
color gray80, all

select R_hot_spots, resi 47+49+50+134+135+136+137+138+139+140+141+142+143+176 and chain A
color firebrick, R_hot_spots

select S_hot_spots, resi 26+29+30+36+38+39+42 and chain A
color marine, S_hot_spots



In [75]:
print("select catalytic_triad, resi 8+96+138 and chain A")
print("color magenta, catalytic_triad\n")

print("select iron_coordinating, resi 49+51+57+71 and chain A")
print("color cyan, iron_coordinating")

select catalytic_triad, resi 8+96+138 and chain A
color magenta, catalytic_triad

select iron_coordinating, resi 49+51+57+71 and chain A
color cyan, iron_coordinating


In [16]:
print_pymol_selection_commands(df_ethA)

select all
color gray80, all

select R_hot_spots, resi 55+57+59+63+293 and chain A
color firebrick, R_hot_spots



In [77]:
print_pymol_selection_commands(df_katG, chain_name_lst=['A', 'B'])

select all
color gray80, all

select R_hot_spots, resi 84+86+87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+120+121+122+123+124+125+126+127+128+129+132+133+134+135+136+137+138+139+140+141+142+143+144+145+146+147+148+149+161+162+165+166+190+228+231+232+233+265+273+274+275+276+277+278+284+288+297+298+299+300+301+302+307+308+309+310+311+312+313+314+315+316+317+326+367+368+418+420 and chain A
color firebrick, R_hot_spots

select R_hot_spots, resi 84+86+87+88+89+90+91+92+93+94+95+96+97+98+99+100+101+102+103+104+105+106+107+108+109+110+111+112+113+120+121+122+123+124+125+126+127+128+129+132+133+134+135+136+137+138+139+140+141+142+143+144+145+146+147+148+149+161+162+165+166+190+228+231+232+233+265+273+274+275+276+277+278+284+288+297+298+299+300+301+302+307+308+309+310+311+312+313+314+315+316+317+326+367+368+418+420 and chain B
color firebrick, R_hot_spots

select S_hot_spots, resi 445+446+447+448+449+450+455+456+458+460+466+467+468+469+504+506+5

In [31]:
# A/C
print_pymol_selection_commands(df_gyrA, chain_name_lst=['A', 'C'], name_prefix='gyrA_')

select all
color gray80, all

select gyrA_R_hot_spots, resi 51+52+55+73+74+75+76+77+78+79+80+81+82+83+84+85+86+87+88+89+90+91+92+93+94+95+96+97+98+99+116+123+124+125+126+127+128+130 and chain A
color firebrick, gyrA_R_hot_spots

select gyrA_R_hot_spots, resi 51+52+55+73+74+75+76+77+78+79+80+81+82+83+84+85+86+87+88+89+90+91+92+93+94+95+96+97+98+99+116+123+124+125+126+127+128+130 and chain C
color firebrick, gyrA_R_hot_spots

select gyrA_S_hot_spots, resi 169+170+171+192+193+194+195+196+197+202+203+204+205+207+208+209+210+211+212+214+215+216+217+219+221+226+237+240+241+244+245+357+364+365+366+367+368+369+370+371+372+373+374+375+376+377+378+382+402+406+409+428+429+451+452+453+454+455+456+457+458+459+460+461+462+463+464+465+466+467+468+469+470+471+472+473+474+475+479+481+482+483+485+486 and chain A
color marine, gyrA_S_hot_spots

select gyrA_S_hot_spots, resi 169+170+171+192+193+194+195+196+197+202+203+204+205+207+208+209+210+211+212+214+215+216+217+219+221+226+237+240+241+244+245+357+364+

In [32]:
# select gyrB_R_hot_spots, (resi 498+499+500+501+502+503+504+505 and chain B) or (resi 498+499+500+501+502+503+504+505 and chain D)
# select gyrB_S_hot_spots, (resi 674+675 and chain B) or (resi 674+675 and chain D)
print_pymol_selection_commands(df_gyrB, chain_name_lst=['B', 'D'], name_prefix='gyrB_')

select all
color gray80, all

select gyrB_R_hot_spots, resi 439+481+482+485+486+495+496+497+498+499+500+501+502+503+504+505+506 and chain B
color firebrick, gyrB_R_hot_spots

select gyrB_R_hot_spots, resi 439+481+482+485+486+495+496+497+498+499+500+501+502+503+504+505+506 and chain D
color firebrick, gyrB_R_hot_spots

select gyrB_S_hot_spots, resi 568+569+585+586+587+588+590+591+593+594+595+596+598+599+600+601+602+603+604+605+606+607+608+609+630+632+633+634 and chain B
color marine, gyrB_S_hot_spots

select gyrB_S_hot_spots, resi 568+569+585+586+587+588+590+591+593+594+595+596+598+599+600+601+602+603+604+605+606+607+608+609+630+632+633+634 and chain D
color marine, gyrB_S_hot_spots



In [58]:
print_pymol_selection_commands(df_rpoB, coord_adj=6, chain_name_lst=['C'])

select all
color gray80, all

select R_hot_spots, resi 53+54+55+56+57+58+160+161+163+164+165+166+167+168+171+172+173+174+175+176+177+178+179+180+181+376+377+378+379+380+381+382+383+384+385+430+431+432+433+434+435+436+437+438+439+440+441+442+443+444+445+446+447+448+449+450+451+452+453+454+455+456+457+458+459+460+461+462+463+464+465+466+468+472+473+482+483+484+485+486+487+488+489+490+491+492+493+494+495+496+497+498+499+500+501+502+505+506+588+589+590+591+609+610+611+612+613+614+615+634+637+638+639+640+678+679+680 and chain C
color firebrick, R_hot_spots



# Combine all GeO score files into a single Excel filr for the supplement

In [2]:
files_to_combine = glob.glob("./supplement/*_GeO_scores.csv")

In [8]:
df_combined = []

for fName in files_to_combine:
    df = pd.read_csv(fName)
    df['protein'] = os.path.basename(fName).split('_')[0]
    df_combined.append(df)

In [13]:
with pd.ExcelWriter('supplement/Final/Supplementary_Data_7.xlsx', engine='openpyxl') as writer:
    for df in df_combined:
        df.to_excel(writer, sheet_name=df['protein'].values[0], index=False)